In [2]:
import pandas as pd
import glob

In [8]:
# Iterate through all sheets in the excel file and concatenate them into a single dataframe
flight_sheet_paths = glob.glob("../0. Data/1 Year Flight Data/A3*.xlsx")
dfs = []
for sheet_path in flight_sheet_paths:
    df = pd.concat(pd.read_excel(sheet_path, sheet_name=None))
    df = df[['DATE', 'FROM', 'TO', 'FLIGHT', 'FLIGHT TIME', 'STATUS']].dropna(subset=['FLIGHT'])
    df['AIRCRAFT REGISTER'] = df.index
    df['AIRCRAFT REGISTRATION'] = df['AIRCRAFT REGISTER'].apply(lambda x: x[0])
    df['AIRCRAFT TYPE'] = sheet_path.split("/")[-1].split(" ")[0]
    print(df['AIRCRAFT TYPE'].iloc[0])
    dfs.append(df)

flights_df = pd.concat(dfs).reset_index(drop=True)

A321
A320Neo


/var/folders/qq/m6bmnsqd617gjhfgc8tzfbhh0000gn/T/ipykernel_80654/2507112207.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(pd.read_excel(sheet_path, sheet_name=None))


A321Neo
A320
A319


In [10]:
# transform flight time column to string
flights_df['FLIGHT TIME'] = flights_df['FLIGHT TIME'].astype(str)
# mark as missing the rows where flight time is not a number
flights_df.loc[~flights_df['FLIGHT TIME'].str.contains(r'\d'), 'FLIGHT TIME'] = None
# convert flight time to minutes assuming format is HH:MM:SS
flights_df['FLIGHT TIME MINUTES'] = pd.to_timedelta(flights_df['FLIGHT TIME']).dt.total_seconds() / 60

In [11]:
# 1. Group by Route
group = flights_df.groupby(["FROM", "TO"])["FLIGHT TIME MINUTES"]

# 2. Calculate the bounds (broadcasted to the original dataframe shape)
m = group.transform("mean")
s = group.transform("std")

# 3. Filter using a boolean mask
# We keep rows within 3 standard deviations OR rows that are NaN
mask = (flights_df["FLIGHT TIME MINUTES"].between(m - 3*s, m + 3*s)) | (flights_df["FLIGHT TIME MINUTES"].isna())

flights_df = flights_df[mask].copy()

In [12]:
# 1. Calculate the mean for each group and broadcast it to the original shape
group_means = flights_df.groupby(['FROM', 'TO', 'AIRCRAFT TYPE'])['FLIGHT TIME MINUTES'].transform('mean').round(2)
# 2. Fill only the NaNs using those calculated means
flights_df['FLIGHT TIME MINUTES'] = flights_df['FLIGHT TIME MINUTES'].fillna(group_means)

# 3. Reassign pending nans with less granular means (only FROM and TO)
group_means = flights_df.groupby(['FROM', 'TO'])['FLIGHT TIME MINUTES'].transform('mean').round(2)
flights_df['FLIGHT TIME MINUTES'] = flights_df['FLIGHT TIME MINUTES'].fillna(group_means)
flights_df.dropna(subset=['FLIGHT TIME MINUTES'], inplace=True)

flights_df['DATE'] = pd.to_datetime(flights_df['DATE'])
flights_df['DATE'] = flights_df['DATE'].dt.strftime('%Y-%m-%d')
flights_df = flights_df[["DATE","FROM", "TO", "FLIGHT", "FLIGHT TIME MINUTES", "AIRCRAFT REGISTRATION", "AIRCRAFT TYPE"]]

In [13]:
flights_df.__len__()

222725

In [14]:
# extract IATA codes from FROM and TO columns and merge with airports data to get lat/lon (ultimately to calculate distance)
airports = pd.read_csv("../0. Data/airports.csv")[['iata_code','name', 'latitude_deg', 'longitude_deg']].dropna(subset=['iata_code'])

flights_df['FROM_IATA'] = flights_df['FROM'].str.extract(r'\(([A-Z]{3})\)')
flights_df['TO_IATA'] = flights_df['TO'].str.extract(r'\(([A-Z]{3})\)')
flights_df = flights_df.merge(airports.rename(columns={'iata_code': 'FROM_IATA', 'name': 'FROM_NAME', 'latitude_deg': 'FROM_LAT', 'longitude_deg': 'FROM_LON'}), on='FROM_IATA', how='left')
flights_df = flights_df.merge(airports.rename(columns={'iata_code': 'TO_IATA', 'name': 'TO_NAME', 'latitude_deg': 'TO_LAT', 'longitude_deg': 'TO_LON'}), on='TO_IATA', how='left')

In [15]:
# approximate the distance from origin to destination using the geodesic distance
from geopy.distance import geodesic
def calculate_distance(row):
    origin = (row['FROM_LAT'], row['FROM_LON'])
    destination = (row['TO_LAT'], row['TO_LON'])
    return geodesic(origin, destination).km

flights_df['DISTANCE_KM'] = flights_df.apply(calculate_distance, axis=1).round(2)


In [16]:
flights_df = flights_df[
    [
        "DATE",
        "FROM_IATA",
        "TO_IATA",
        "AIRCRAFT TYPE",
        "FLIGHT TIME MINUTES",
        "DISTANCE_KM",
    ]
].rename(
    columns={
        "AIRCRAFT TYPE": "aircraft_type_icao",
        "FLIGHT TIME MINUTES": "flight_time_minutes",
        "DISTANCE_KM": "total_distance_km",
    }
)

# Replace A-XXX with AXXX in the aircraft_type_icao column
flights_df['aircraft_type_icao'] = flights_df['aircraft_type_icao'].str.replace('-', '')

---

## Emissions Calculation

### Source: 1.A.3.a Aviation - Annex 5 - Master emissions calculator 2019

In [17]:
import pandas as pd
import numpy as np

def quantify_vueling_emissions(df):
    """
    Quantify CO2 emissions using EEA/EMEP 2019 Tier 2 Aviation methodology.
    Coefficients derived directly from the EEA Master Emissions Calculator 2019
    (1_A_3_a_Aviation_Annex_5), 'database + table pivot' sheet.

    LTO CO2: from the ICAO default LTO cycle (single airport, FORECAST col F).
    Source cells in 'database + table pivot' sheet (col F = FORECAST CO2):
        A319 LTO CO2 = 2169.76 kg → MATCH(AF="A319" & AM="LTO",  AO=125) → col F
        A320 LTO CO2 = 2570.93 kg → MATCH(AF="A320" & AM="LTO",  AO=125) → col F
        A321 LTO CO2 = 2954.91 kg → MATCH(AF="A321" & AM="LTO-2", AO=200) → col F

    CCD fuel coefficients: OLS linear fit over the 10 stage-length breakpoints
        (125, 200, 250, 500, 750, 1000, 1500, 2000, 2500, 3000 NM).
    Source cells: MATCH(AF="A3xx" & AM="CCD", AO=<stage>) → col E (FORECAST FUEL BURNT KG)
    CCD CO2 = CCD fuel * 3.15 (kerosene emission factor).
    NEO adjustment: -15% efficiency gain applied to both LTO and CCD.
    """

    # Stage-length lookup table from the EEA spreadsheet (NM → fuel kg)
    # Col E = FORECAST FUEL BURNT KG, keyed by AF (IMPACT ACFT ID) + AM (LTO/CCD) + AO (stage)
    CCD_TABLES = {
        'A319': {
            # AF="A319", AM="CCD", AO=stage → col E (FORECAST FUEL BURNT KG)
            'stages': [125, 200, 250, 500, 750, 1000, 1500, 2000, 2500, 3000],
            'fuel':   [903.68, 1318.23, 1600.25, 2844.14, 3884.66,
                       4875.32, 7145.63, 9493.24, 11713.77, 14663.73],
            # AF="A319", AM="LTO", AO=125 → col F (FORECAST CO2): 2169.76 kg
            'lto_co2': 2169.76,
        },
        'A320': {
            # AF="A320", AM="CCD", AO=stage → col E (FORECAST FUEL BURNT KG)
            'stages': [125, 200, 250, 500, 750, 1000, 1500, 2000, 2500, 3000],
            'fuel':   [931.92, 1356.45, 1647.38, 2946.00, 4124.49,
                       5273.37, 7768.61, 10483.84, 12914.24, 15846.86],
            # AF="A320", AM="LTO", AO=125 → col F (FORECAST CO2): 2570.93 kg
            'lto_co2': 2570.93,
        },
        'A321': {
            # AF="A321", AM="CCD", AO=stage → col E (FORECAST FUEL BURNT KG)
            'stages': [125, 200, 250, 500, 750, 1000, 1500, 2000, 2500],
            'fuel':   [1142.71, 1675.97, 2031.55, 3642.21, 5132.33,
                       6586.88, 9724.64, 13078.11, 16103.98],
            # AF="A321", AM="LTO-2", AO=200 → col F (FORECAST CO2): 2954.91 kg
            # (LTO-2 = busy airport default, used instead of ICAO LTO for A321)
            'lto_co2': 2954.91,
        },
    }

    LINEAR_COEFFS = {}
    for ac, d in CCD_TABLES.items():
        b, a = np.polyfit(d['stages'], d['fuel'], 1)
        LINEAR_COEFFS[ac] = (a, b)

    def _interpolate_ccd_fuel(stages, fuels, dist_nm):
        """
        Piecewise linear interpolation matching the Excel FORECAST method:
        clamp to the nearest pair of known stage-length breakpoints.
        """
        stages = np.array(stages)
        fuels  = np.array(fuels)
        if dist_nm <= stages[0]:
            return float(fuels[0])
        if dist_nm >= stages[-1]:
            return float(fuels[-1])
        idx = np.searchsorted(stages, dist_nm) - 1
        x0, x1 = stages[idx], stages[idx + 1]
        y0, y1 = fuels[idx],  fuels[idx + 1]
        return float(y0 + (y1 - y0) * (dist_nm - x0) / (x1 - x0))

    def calculate_row(row):
        # ── 1. Unit conversion ────────────────────────────────────────────
        dist_nm = row['total_distance_km'] * 0.539957
        ac_type = str(row['aircraft_type_icao']).upper()

        # ── 2. Identify base aircraft family ─────────────────────────────
        if 'A321' in ac_type:
            family = 'A321'
        elif 'A319' in ac_type:
            family = 'A319'
        else:
            family = 'A320'          # default / A320 baseline

        table = CCD_TABLES[family]

        # ── 3. LTO CO2 (kg) ──────────────────────────────────────────────
        lto_co2 = table['lto_co2']

        # ── 4. CCD fuel via piecewise interpolation (matches Excel FORECAST)
        ccd_fuel = _interpolate_ccd_fuel(table['stages'], table['fuel'], dist_nm)
        ccd_co2  = ccd_fuel * 3.15

        # ── 5. NEO efficiency adjustment (–15 %) ─────────────────────────
        if 'N' in ac_type:           # e.g. A320N, A321NX, A319N
            lto_co2 *= 0.85
            ccd_co2 *= 0.85

        return pd.Series({
            'LTO_CO2_kg':   round(lto_co2,           2),
            'CCD_CO2_kg':   round(ccd_co2,            2),
            'Total_CO2_kg': round(lto_co2 + ccd_co2, 2),
        })

    emission_results = df.apply(calculate_row, axis=1)
    return pd.concat([df, emission_results], axis=1)

| Aircraft | LTO CO₂ (kg) | CCD a (fuel intercept, kg) | CCD b (fuel slope, kg/NM) |
|---|---|---|---|
| A319 | 2169.76 | 354.59 | 4.6424 |
| A320 | 2570.93 | 291.13 | 5.1063 |
| A321 | 2954.91 | 411.42 | 6.2794 |

In [18]:
flights_df_emissions = quantify_vueling_emissions(flights_df)

In [19]:
flights_df_emissions

,DATE,FROM_IATA,TO_IATA,aircraft_type_icao,flight_time_minutes,total_distance_km,LTO_CO2_kg,CCD_CO2_kg,Total_CO2_kg
0,2026-02-02,MUC,BCN,A321,104.40,1095.78,2954.91,13194.19,16149.10
1,2026-02-02,BCN,MUC,A321,100.99,1095.78,2954.91,13194.19,16149.10
2,2026-02-02,OPO,BCN,A321,80.80,900.98,2954.91,11198.79,14153.70
3,2026-02-02,BCN,OPO,A321,86.57,900.98,2954.91,11198.79,14153.70
4,2026-02-02,CDG,BCN,A321,84.00,857.87,2954.91,10726.39,13681.30
...,...,...,...,...,...,...,...,...,...
222720,2025-02-03,FLR,ORY,A319,81.00,871.46,2169.76,8497.49,10667.25
222721,2025-02-02,LGW,FLR,A319,104.00,1181.91,2169.76,10770.66,12940.42
222722,2025-02-02,FLR,LGW,A319,98.00,1181.91,2169.76,10770.66,12940.42
222723,2025-02-02,BCN,FLR,A319,77.00,799.09,2169.76,7885.04,10054.80


In [21]:
flights_df.to_csv('../0. Data/vueling_flights_feb25-feb26.csv', index=False)

In [ ]:
flights_df_emissions.to_csv("../0. Data/1 Year Flight Data/fr24_vueling_emissions_feb25-feb26.csv", index=False)